In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, Model
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix
import matplotlib.pyplot as plt

# 1. Load ECG dataset directly via URL
dataframe = pd.read_csv('http://storage.googleapis.com/download.tensorflow.org/data/ecg.csv', header=None)
raw_data = dataframe.values
labels = raw_data[:, -1]
data = raw_data[:, 0:-1]

train_data, test_data, train_labels, test_labels = train_test_split(
    data, labels, test_size=0.2, random_state=21
)

# 2. Normalize data to [0, 1]
min_val = tf.reduce_min(train_data)
max_val = tf.reduce_max(train_data)
train_data = tf.cast((train_data - min_val) / (max_val - min_val), tf.float32)
test_data = tf.cast((test_data - min_val) / (max_val - min_val), tf.float32)

# Filter only normal rhythms (label == 1) for training
train_labels = train_labels.astype(bool)
test_labels = test_labels.astype(bool)
normal_train_data = train_data[train_labels]
normal_test_data = test_data[test_labels]
anomalous_test_data = test_data[~test_labels]

# 3. Build Encoder-Decoder Autoencoder Architecture
class AnomalyDetector(Model):
    def __init__(self):
        super(AnomalyDetector, self).__init__()
        self.encoder = tf.keras.Sequential([
            layers.Dense(32, activation="relu"),
            layers.Dense(16, activation="relu"),
            layers.Dense(8, activation="relu")
        ])
        self.decoder = tf.keras.Sequential([
            layers.Dense(16, activation="relu"),
            layers.Dense(32, activation="relu"),
            layers.Dense(140, activation="sigmoid")
        ])

    def call(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded

autoencoder = AnomalyDetector()
autoencoder.compile(optimizer='adam', loss='mae')

# 4. Train ONLY on normal data
history = autoencoder.fit(
    normal_train_data, normal_train_data,
    epochs=20, batch_size=512,
    validation_data=(test_data, test_data),
    shuffle=True
)

# 5. Calculate reconstruction loss and set threshold
reconstructions = autoencoder.predict(normal_train_data)
train_loss = tf.keras.losses.mae(reconstructions, normal_train_data)
threshold = np.mean(train_loss) + np.std(train_loss)
print("\nAnomaly Threshold:", threshold)

# 6. Predict on Test Data
test_reconstructions = autoencoder.predict(test_data)
test_loss = tf.keras.losses.mae(test_reconstructions, test_data)
preds = tf.math.less(test_loss, threshold)

print("Test Accuracy:", accuracy_score(test_labels, preds))
print("Confusion Matrix:\n", confusion_matrix(test_labels, preds))

Epoch 1/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 4s 154ms/step - loss: 0.0581 - val_loss: 0.0533
Epoch 2/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 44ms/step - loss: 0.0558 - val_loss: 0.0514
Epoch 3/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - loss: 0.0526 - val_loss: 0.0498
Epoch 4/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 92ms/step - loss: 0.0488 - val_loss: 0.0477
Epoch 5/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step - loss: 0.0447 - val_loss: 0.0454
Epoch 6/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - loss: 0.0407 - val_loss: 0.0434
Epoch 7/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step - loss: 0.0372 - val_loss: 0.0419
Epoch 8/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 102ms/step - loss: 0.0342 - val_loss: 0.0407
Epoch 9/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step - loss: 0.0317 - val_loss: 0.0394
Epoch 10/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step - loss: 0.0296 - val_loss: 0.0385
Epoch 11/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 72ms/step - loss: 0.0279 - val_loss: 0.0376
Epoch 12/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step - loss: 0.0265 - val_loss: 0.0368